# 007 - XGBoost

## Objetivo

Avaliar XGBoost na previsão de `AGE_RANGE` (16 classes), mantendo as mesmas variáveis e a divisão estratificada 80/20 do notebook 006. A seleção de hiperparâmetros usa Macro F1 média em três folds apenas na parcela de treino; a validação fixa fica reservada para uma avaliação final.

Referências na validação fixa: regressão logística regularizada do 002 (**0,4125**), árvore ajustada do 005 (**0,4037**) e Random Forest do 006 (**0,3692**). Este notebook não gera submissão.

## Ambiente, dados e divisão fixa

Reutilizamos `preparar_dados()`, `make_stratified_split()` e `make_model_pipeline()` do módulo comum. A divisão é `test_size=0.2`, `random_state=42`, tal como no 006. O pré-processador pertence à pipeline e, por isso, é reajustado apenas nos dados de treino de cada fold e, depois da seleção, em todo `X_train` antes de prever a validação fixa. O ambiente precisa do pacote `xgboost` (por exemplo, `python -m pip install xgboost`) antes de executar a experiência.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import sklearn
import xgboost

candidate_roots = [Path.cwd(), *Path.cwd().resolve().parents]
repo_root = next(
    (p for p in candidate_roots if (p / 'notebooks').is_dir() and (p / 'src' / 'preparar_dados.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto com notebooks/ e src/preparar_dados.py.')

sys.path.insert(0, str(repo_root))
from src.preparar_dados import make_model_pipeline, make_stratified_split, preparar_dados

train_path = repo_root / 'notebooks' / 'dataset_train.csv'
test_path = repo_root / 'notebooks' / 'dataset_test_X.csv'
if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(f'CSV de treino/teste não encontrados em {repo_root / "notebooks"}')

print(f'Python: {sys.version}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'XGBoost: {xgboost.__version__}')
print(f'Raiz do projeto: {repo_root}')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
prepared = preparar_dados(train_df, test_df)
X_train, X_val, y_train, y_val = make_stratified_split(
    prepared['X_train'], prepared['y_train'], test_size=0.2, random_state=42
)

assert len(X_train) == 8769 and len(X_val) == 2193, (len(X_train), len(X_val))
assert X_train.columns.equals(X_val.columns), 'As colunas de treino e validação não estão alinhadas.'
assert sorted(y_train.unique().tolist()) == list(range(16)), 'Esperavam-se as 16 classes no treino.'
print(f'Treino: {X_train.shape}; validação: {X_val.shape}')
print(f'Classes no alvo: {sorted(y_train.unique().tolist())}')

Python: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
scikit-learn: 1.6.1
XGBoost: 3.4.1
Raiz do projeto: /Users/eduardaferreira/daa-kaggle-2026-2027


Treino: (8769, 236); validação: (2193, 236)
Classes no alvo: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]


## Pesquisa de hiperparâmetros no treino

Testamos as quatro combinações de `max_depth=[3, 5]` e `learning_rate=[0.05, 0.1]`, mantendo `objective='multi:softprob'`, `tree_method='hist'`, `n_estimators=200`, `random_state=42` e `n_jobs=1`. A CV usa três folds estratificados (`shuffle=True`, `random_state=42`), `scoring='f1_macro'` e `n_jobs=1`. A pipeline completa garante que imputação e codificação são aprendidas dentro de cada fold.

In [2]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

xgb_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
xgb_search = GridSearchCV(
    estimator=make_model_pipeline(
        XGBClassifier(
            objective='multi:softprob',
            num_class=16,
            tree_method='hist',
            n_estimators=200,
            random_state=42,
            n_jobs=1,
            verbosity=0,
        ),
        normalize_numeric=False,
    ),
    param_grid={
        'model__max_depth': [3, 5],
        'model__learning_rate': [0.05, 0.1],
    },
    scoring='f1_macro',
    cv=xgb_cv,
    n_jobs=1,
    refit=True,
    return_train_score=True,
    error_score='raise',
    verbose=1,
)
xgb_search.fit(X_train, y_train)

xgb_results = pd.DataFrame(xgb_search.cv_results_)
xgb_table = xgb_results[[
    'param_model__max_depth',
    'param_model__learning_rate',
    'mean_train_score',
    'mean_test_score',
    'std_test_score',
    'rank_test_score',
]].rename(columns={
    'param_model__max_depth': 'max_depth',
    'param_model__learning_rate': 'learning_rate',
    'mean_train_score': 'treino_macro_f1_medio',
    'mean_test_score': 'cv_macro_f1_medio',
    'std_test_score': 'cv_macro_f1_desvio_padrao',
    'rank_test_score': 'posicao',
}).sort_values(['posicao', 'max_depth', 'learning_rate'])
display(xgb_table.reset_index(drop=True))

best_params = {
    'max_depth': xgb_search.best_params_['model__max_depth'],
    'learning_rate': xgb_search.best_params_['model__learning_rate'],
}
best_cv_macro_f1 = xgb_search.best_score_
print(f'Melhores parâmetros por CV: {best_params}')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')

Fitting 3 folds for each of 4 candidates, totalling 12 fits


,max_depth,learning_rate,treino_macro_f1_medio,cv_macro_f1_medio,cv_macro_f1_desvio_padrao,posicao
0,5,0.10,0.991276,0.444887,0.004273,1
1,3,0.10,0.828632,0.441817,0.001524,2
2,5,0.05,0.914918,0.440933,0.003942,3
3,3,0.05,0.667137,0.436111,0.000887,4


Melhores parâmetros por CV: {'max_depth': 5, 'learning_rate': 0.1}
Macro F1 CV média: 0.4449


## Avaliação na validação fixa

A melhor pipeline, reajustada pela pesquisa em todo `X_train`, prevê agora `X_val` sem novo ajuste. Mostramos Macro F1, F1 por classe para as 16 classes e uma matriz de confusão normalizada por linha (proporção de cada classe real distribuída pelas classes previstas). Comparamos a Macro F1 com as referências 002, 005 e 006.

In [3]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score

best_xgb_pipe = xgb_search.best_estimator_
xgb_val_pred = best_xgb_pipe.predict(X_val)
xgb_val_macro_f1 = f1_score(
    y_val, xgb_val_pred, labels=list(range(16)), average='macro', zero_division=0
)
references = {
    '002 - regressão logística': 0.4125,
    '005 - árvore ajustada': 0.4037,
    '006 - Random Forest': 0.3692,
}

print(f'Configuração escolhida: {best_params}')
print('Parâmetros fixos: objective=multi:softprob, tree_method=hist, n_estimators=200, random_state=42, n_jobs=1')
print(f'Macro F1 CV média: {best_cv_macro_f1:.4f}')
print(f'Macro F1 validação: {xgb_val_macro_f1:.4f}')
for name, score in references.items():
    print(f'Diferença vs. {name} ({score:.4f}): {xgb_val_macro_f1 - score:+.4f}')

print('--- F1 por classe na validação ---')
print(classification_report(
    y_val,
    xgb_val_pred,
    labels=list(range(16)),
    zero_division=0,
    target_names=[str(i) for i in range(16)],
))
normalized_confusion = confusion_matrix(
    y_val, xgb_val_pred, labels=list(range(16)), normalize='true'
)
display(pd.DataFrame(
    np.round(normalized_confusion, 3),
    index=pd.Index(range(16), name='real'),
    columns=pd.Index(range(16), name='prevista'),
))

Configuração escolhida: {'max_depth': 5, 'learning_rate': 0.1}
Parâmetros fixos: objective=multi:softprob, tree_method=hist, n_estimators=200, random_state=42, n_jobs=1
Macro F1 CV média: 0.4449
Macro F1 validação: 0.4491
Diferença vs. 002 - regressão logística (0.4125): +0.0366
Diferença vs. 005 - árvore ajustada (0.4037): +0.0454
Diferença vs. 006 - Random Forest (0.3692): +0.0799
--- F1 por classe na validação ---
              precision    recall  f1-score   support

           0       0.78      0.85      0.81        41
           1       0.47      0.24      0.31        34
           2       0.57      0.65      0.61        80
           3       0.39      0.28      0.33        75
           4       0.34      0.24      0.28        90
           5       0.30      0.33      0.31       129
           6       0.35      0.38      0.37       172
           7       0.38      0.38      0.38       173
           8       0.59      0.65      0.62       181
           9       0.60      0.63     

prevista,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
real,,,,,,,,,,,,,,,,
0,0.854,0.024,0.049,0.049,0.000,0.000,0.000,0.000,0.024,0.000,0.000,0.000,0.000,0.000,0.000,0.000
1,0.235,0.235,0.529,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,0.025,0.088,0.650,0.125,0.038,0.025,0.025,0.012,0.012,0.000,0.000,0.000,0.000,0.000,0.000,0.000
3,0.000,0.013,0.107,0.280,0.160,0.173,0.200,0.067,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
4,0.000,0.000,0.067,0.122,0.244,0.278,0.133,0.111,0.044,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,0.000,0.000,0.016,0.031,0.147,0.333,0.349,0.070,0.054,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,0.000,0.000,0.012,0.023,0.017,0.233,0.384,0.238,0.093,0.000,0.000,0.000,0.000,0.000,0.000,0.000
7,0.000,0.000,0.000,0.012,0.017,0.087,0.208,0.382,0.295,0.000,0.000,0.000,0.000,0.000,0.000,0.000
8,0.000,0.000,0.006,0.000,0.017,0.039,0.061,0.232,0.646,0.000,0.000,0.000,0.000,0.000,0.000,0.000


## Conclusão e decisão
- Data: 9/11
- A configuração escolhida foi `max_depth=5`, `learning_rate=0.1`, com `n_estimators=200`. Obteve Macro F1 média CV **0,4449** e Macro F1 **0,4491** na validação fixa. Este resultado supera o 002 (**0,4125**, +0,0366), o 005 (**0,4037**, +0,0454) e o 006 (**0,3692**, +0,0799). 
- Pontuação Kaggle: 0.45
- Conclusões : Overfitting, os escalões 30–34 e 75–79 anos tiveram o F1 mais baixo (0,28).
- Melhorias : Testar min_child_weight mais elevado e ajustar subsample e colsample_bytree para reduzir o sobreajustamento, usar early stopping dentro de cada fold para escolher o número de árvores, mantendo a validação fixa reservada para a avaliação final, comparar o XGBoost com e sem as quatro perguntas condicionais para ver se mudam os indivíduos sinalizados como anómalos.